In [1]:
# import sys
# from pathlib import Path

# # Ищем корень проекта (папку с requirements.txt) вверх по дереву
# root = Path.cwd()
# while root != root.parent and not (root / "requirements.txt").is_file():
#     root = root.parent

# req = root / "requirements.txt"
# print("requirements.txt:", req)

# if req.is_file():
#     !{sys.executable} -m pip install -r "{req}"
# else:
#     print("requirements.txt не найден")

In [2]:
import json
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

True

In [3]:
"""Ячейка 2 — golden-датасет из 10 эталонных вопросов.

Структура: 7 английских вопросов по 3 темам корпуса (linear_model / tree
/ model_evaluation), 2 русских для проверки мультиязычного retrieval'а,
1 meta-вопрос про `about.md`. Поле `ground_truth_url_keywords` нужно для
Recall@k в Ячейке 4: считаем hit, если URL retrieved-чанка содержит
ожидаемое ключевое слово.
"""

GOLDEN = [
    # --- 7 EN in-corpus вопросов ---
    {
        "question": "How does Ridge regression handle multicollinearity?",
        "ground_truth": "Ridge adds an L2 penalty alpha * sum(w_i^2) to the loss, "
        "which shrinks correlated coefficients toward each other.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does the alpha parameter control in Ridge?",
        "ground_truth": "Alpha controls regularization strength; larger alpha means "
        "stronger penalty and smaller coefficients.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What is the difference between Lasso and Ridge?",
        "ground_truth": "Lasso uses L1 penalty which can zero out coefficients (feature "
        "selection); Ridge uses L2 which shrinks but never zeroes.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does min_samples_leaf control in a decision tree?",
        "ground_truth": "min_samples_leaf is the minimum number of samples required to be "
        "at a leaf node; higher values prevent overfitting by limiting depth.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "When does a decision tree overfit?",
        "ground_truth": "Trees overfit when grown too deep without min_samples_leaf or "
        "min_samples_split constraints, memorising training noise.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "What is the formula for precision?",
        "ground_truth": "precision = TP / (TP + FP). Fraction of positive predictions that "
        "are actually positive.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    {
        "question": "When is recall more important than precision?",
        "ground_truth": "Recall matters most when missing positives is costly: cancer "
        "screening, fraud detection, anything where false negatives are "
        "more harmful than false positives.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 2 RU вопроса (тест мультиязычного retrieval) ---
    {
        "question": "Что такое L2-регуляризация?",
        "ground_truth": "L2-регуляризация добавляет к функции потерь штраф, "
        "пропорциональный сумме квадратов коэффициентов модели. "
        "Используется в Ridge.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "Что такое F1-мера?",
        "ground_truth": "F1 — гармоническое среднее precision и recall, "
        "F1 = 2 * precision * recall / (precision + recall).",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 1 meta-вопрос (тест about.md) ---
    {
        "question": "Что ты умеешь?",
        "ground_truth": "Отвечаю на вопросы по трём разделам scikit-learn: линейные "
        "модели, деревья решений, метрики качества.",
        "ground_truth_url_keywords": ["about.md", "local"],
    },
]

In [4]:
import sys
from pathlib import Path

# Ищем корень проекта (папку, где лежит app/) вверх по дереву от cwd
root = Path.cwd()
while root != root.parent and not (root / "app").is_dir():
    root = root.parent

if (root / "app").is_dir() and str(root) not in sys.path:
    sys.path.insert(0, str(root))
    print(f"Корень проекта добавлен в sys.path: {root}")
else:
    print(f"Текущий cwd: {Path.cwd()}")

Корень проекта добавлен в sys.path: d:\агенты боты и воркфлоу\Agents\mentoring\RAG-documentation-sklearn


In [6]:
"""Ячейка 3 — прямое подключение к RAG-pipeline без HTTP-слоя.

`build_rag_chain()` поднимает retriever (Qdrant + embedder) и LCEL-цепочку
в этом же процессе. Прямой вызов даёт чистые тайминги и доступ к
`retrieved_contexts`, которые понадобятся RAGAS в Ячейке 5.
"""

from app.config import settings
from app.rag.chain import build_rag_chain

chain, retriever = build_rag_chain()

c:\Users\akaps\anaconda3\envs\task-api\Lib\site-packages\qdrant_client\qdrant_remote.py:287: UserWarning: Qdrant client version 1.20.0 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2556.62it/s]


In [7]:
"""Ячейка 4 — Recall@k для retriever'а.

Прогоняем каждый golden-вопрос через retriever, забираем URL'ы топ-k чанков
и считаем hit по keyword-совпадению. Дополнительно копим
`retrieved_contexts` — они уйдут в RAGAS на следующем шаге.
"""


def url_match(returned_urls: list[str], expected_keywords: list[str]) -> bool:
    """Помечает retrieval как hit, если хоть один URL содержит ожидаемое ключевое слово.

    Используется в Recall@k: keyword-matching на уровне модуля sklearn
    (`linear_model` / `tree` / `model_evaluation`) — этого достаточно
    для понимания «не промазал ли retriever мимо темы целиком».
    """
    if not expected_keywords:  # ловушка под OOC-вопросы, если решите добавлять
        return not returned_urls
    return any(
        any(kw.lower() in url.lower() for kw in expected_keywords)
        for url in returned_urls
    )


results = []
for item in GOLDEN:
    docs = retriever.invoke(item["question"])
    urls = [doc.metadata.get("source", "") for doc in docs]
    results.append(
        {
            "question": item["question"],
            "ground_truth": item["ground_truth"],
            "retrieved_urls": urls,
            "retrieved_contexts": [doc.page_content for doc in docs],
            "retriever_hit": url_match(urls, item["ground_truth_url_keywords"]),
        }
    )

hits = sum(1 for r in results if r["retriever_hit"])
recall_at_k = hits / len(results)
print(f"Retriever Recall@{settings.top_k}: {recall_at_k:.3f}  ({hits}/{len(results)})")

Retriever Recall@4: 1.000  (10/10)


In [15]:
"""Ячейка 5 — generation-метрики через RAGAS (режим бесплатных лимитов).

RAGAS 0.2.x требует Dataset со строго заданными именами полей
(`user_input` / `response` / `retrieved_contexts` / `reference`). LLM
и эмбеддер оборачиваем в `LangchainLLMWrapper` и
`LangchainEmbeddingsWrapper` — без этого `evaluate()` не поймёт, как
через них ходить. Считаем Faithfulness (опора на контекст) и
ResponseRelevancy (релевантность ответа вопросу).

У `:free`-моделей OpenRouter лимит ~50 запросов в день, поэтому считаем
порциями по BATCH_SIZE вопросов, а ответы и оценки копим в
`notebooks/ragas_cache.json`. Ячейку можно запускать повторно (хоть
на следующий день): считается только то, чего ещё нет в кэше.
"""

import math

from datasets import Dataset
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from ragas import RunConfig, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy

BATCH_SIZE = 3  # вопросов за один запуск; при 429 уменьшить
METRIC_NAMES = ("faithfulness", "answer_relevancy")

CACHE_PATH = root / "notebooks" / "ragas_cache.json"
cache = (
    json.loads(CACHE_PATH.read_text(encoding="utf-8")) if CACHE_PATH.exists() else {}
)


def save_cache() -> None:
    """Сохранить кэш на диск после каждого шага: прерывание ничего не теряет."""
    CACHE_PATH.write_text(
        json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8"
    )


def is_done(entry: dict) -> bool:
    """Вопрос готов, если есть ответ и обе метрики посчитаны."""
    return "response" in entry and all(m in entry for m in METRIC_NAMES)


# Порция на этот запуск: первые BATCH_SIZE вопросов, которые ещё не готовы.
todo = [r for r in results if not is_done(cache.get(r["question"], {}))][:BATCH_SIZE]

# Шаг 1. Ответы chain — по одному запросу на вопрос, кэшируем.
for r in todo:
    entry = cache.setdefault(r["question"], {})
    if "response" in entry:
        continue
    try:
        entry["response"] = chain.invoke(r["question"])
        save_cache()
    except Exception as exc:  # 429 / 402 / сеть: останавливаемся, прогресс сохранён
        print(f"chain.invoke остановлен на {r['question']!r}: {type(exc).__name__}")
        break

if todo:
    # Судья для RAGAS. На :free-моделях кредиты не тратятся, поэтому
    # max_tokens с запасом: при коротком лимите RAGAS падает с
    # LLMDidNotFinishException. timeout не даёт запросу зависнуть.
    judge = ChatOpenAI(
        base_url=settings.llm_base_url,
        api_key=settings.llm_api_key,
        model=settings.llm_model,
        temperature=settings.llm_temperature,
        max_tokens=4096,
        timeout=120,
    )

    # LLM и эмбеддер для RAGAS оборачиваются в специальные wrapper'ы
    llm = LangchainLLMWrapper(judge)
    emb = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(
            model_name=settings.embedding_model,
            encode_kwargs={"normalize_embeddings": settings.normalize_embeddings},
        )
    )
    metric_objects = {
        "faithfulness": Faithfulness(llm=llm),
        "answer_relevancy": ResponseRelevancy(llm=llm, embeddings=emb),
    }

    # Шаг 2. Каждую метрику считаем только для вопросов, где её ещё нет:
    # не тратим дневной лимит запросов на уже готовые значения.
    for name, metric in metric_objects.items():
        rows = [
            r
            for r in todo
            if "response" in cache[r["question"]] and name not in cache[r["question"]]
        ]
        if not rows:
            continue

        # RAGAS 0.2.x требует именно эти имена полей в Dataset:
        # user_input / response / retrieved_contexts / reference
        ragas_data = Dataset.from_list(
            [
                {
                    "user_input": r["question"],
                    "response": cache[r["question"]]["response"],
                    "retrieved_contexts": r["retrieved_contexts"],
                    "reference": r["ground_truth"],
                }
                for r in rows
            ]
        )
        scores = evaluate(
            dataset=ragas_data,
            metrics=[metric],
            # Лимит free-моделей — 20 запросов/мин: один воркер и повторы
            # с паузой, чтобы не упираться в 429.
            run_config=RunConfig(
                timeout=300, max_workers=1, max_retries=5, max_wait=60
            ),
        )

        # Шаг 3. Сохраняем только реально посчитанные значения (не NaN).
        for _, row in scores.to_pandas().iterrows():
            if not math.isnan(row[name]):
                cache[row["user_input"]][name] = float(row[name])
        save_cache()

done = sum(is_done(e) for e in cache.values())
print(f"Готово вопросов: {done}/{len(GOLDEN)}. Кэш: {CACHE_PATH}")
if done < len(GOLDEN):
    print("Запусти эту ячейку снова (при 429 — после 00:00 UTC), затем Ячейку 6.")

Готово вопросов: 10/10. Кэш: d:\агенты боты и воркфлоу\Agents\mentoring\RAG-documentation-sklearn\notebooks\ragas_cache.json


In [16]:
"""Ячейка 6 — сохраняем сводку метрик в JSON.

Берём mean по каждой RAGAS-метрике из кэша Ячейки 5, добавляем
retriever-Recall@k и метаданные прогона (модель, embedder, top_k).
Файл `rag_metrics.json` цитируется в README
"""

# Файл пишем только когда посчитаны ВСЕ вопросы: среднее по 3 из 10
# нельзя выдавать за оценку всей системы.
missing = [
    g["question"]
    for g in GOLDEN
    if not all(m in cache.get(g["question"], {}) for m in METRIC_NAMES)
]
if missing:
    raise RuntimeError(
        f"Метрики посчитаны не для всех вопросов ({len(GOLDEN) - len(missing)}/"
        f"{len(GOLDEN)}). Запусти Ячейку 5 ещё раз. Не хватает:\n- "
        + "\n- ".join(missing)
    )

gen_scores = {
    m: sum(cache[g["question"]][m] for g in GOLDEN) / len(GOLDEN) for m in METRIC_NAMES
}

metrics = {
    "n_questions": len(GOLDEN),
    "model": settings.llm_model,
    "embedding_model": settings.embedding_model,
    "top_k": settings.top_k,
    "retriever": {
        f"recall_at_{settings.top_k}": recall_at_k,
        "hits": hits,
    },
    "generation": gen_scores,
}

# Путь строим от корня проекта (`root` найден в Ячейке 2), а не от cwd:
# VS Code запускает ноутбук из папки notebooks/, и относительный путь
# "notebooks/rag_metrics.json" превращался в notebooks/notebooks/...
out_path = root / "notebooks" / "rag_metrics.json"
out_path.write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",  # на Windows без этого write_text пишет в cp1251
)
print(f"Метрики сохранены: {out_path}")
print(json.dumps(metrics, indent=2, ensure_ascii=False))

Метрики сохранены: d:\агенты боты и воркфлоу\Agents\mentoring\RAG-documentation-sklearn\notebooks\rag_metrics.json
{
  "n_questions": 10,
  "model": "meta-llama/llama-3.3-70b-instruct",
  "embedding_model": "intfloat/multilingual-e5-small",
  "top_k": 4,
  "retriever": {
    "recall_at_4": 1.0,
    "hits": 10
  },
  "generation": {
    "faithfulness": 0.8812229437229437,
    "answer_relevancy": 0.939668598612589
  }
}
